# Phase 4: Common Research-Paper Preprocessing Pipeline

This notebook combines each paper's `Title` and `Abstract`, then applies conservative NLP preprocessing. Technical terms and acronym tokens are retained, including forms such as BERT, CNN, HPC, GPU, NLP, LLM, GPT-4, and BERT-based.

In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Iterable

import pandas as pd

try:
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
except ImportError as error:
    raise ImportError("Install nltk before running this notebook: pip install nltk") from error

REQUIRED_COLUMNS = [
    "Title",
    "Author",
    "Institution",
    "Topic",
    "Domain",
    "Field",
    "Concept",
    "Abstract",
    "Keyword",
    "Citation count",
    "Cites",
    "Top 1% cited",
    "Top 10% cited",
    "DOI",
]

TECHNICAL_TERMS = {"bert", "cnn", "hpc", "gpu", "nlp", "llm", "rnn", "lstm", "gpt", "api", "cuda", "cpu", "tpu"}

def _default_stopwords() -> set[str]:
    try:
        return set(stopwords.words("english")) - TECHNICAL_TERMS
    except LookupError:
        # A compact fallback keeps the notebook usable before NLTK data is downloaded.
        return {"a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "in", "is", "it", "of", "on", "or", "that", "the", "this", "to", "was", "were", "with"}

STOP_WORDS = _default_stopwords()
LEMMATIZER = WordNetLemmatizer()
TOKEN_PATTERN = re.compile(r"(?u)[a-z0-9]+(?:[-'][a-z0-9]+)*")

def tokenize_text(text: str) -> list[str]:
    """Lowercase text and extract words, numbers, and technical hyphenated terms."""
    if not isinstance(text, str):
        return []
    return TOKEN_PATTERN.findall(text.casefold())

def remove_stopwords(tokens: Iterable[str], stop_words: set[str] | None = None) -> list[str]:
    """Remove general English stopwords without removing technical vocabulary."""
    blocked = STOP_WORDS if stop_words is None else stop_words - TECHNICAL_TERMS
    return [token for token in tokens if token not in blocked]

def _lemmatize(tokens: Iterable[str]) -> list[str]:
    try:
        return [LEMMATIZER.lemmatize(token) for token in tokens]
    except LookupError:
        return list(tokens)

def clean_text(text: str, *, lemmatize: bool = False, stop_words: set[str] | None = None) -> str:
    """Return whitespace-separated clean tokens for one title/abstract string."""
    tokens = remove_stopwords(tokenize_text(text), stop_words)
    if lemmatize:
        tokens = _lemmatize(tokens)
    return " ".join(tokens)

def preprocess_document(title: object, abstract: object, *, lemmatize: bool = False) -> dict[str, str]:
    """Preprocess one paper and return its title, abstract, and clean text."""
    safe_title = "" if pd.isna(title) else str(title).strip()
    safe_abstract = "" if pd.isna(abstract) else str(abstract).strip()
    combined = f"{safe_title} {safe_abstract}".strip()
    return {
        "title": safe_title,
        "abstract": safe_abstract,
        "clean_text": clean_text(combined, lemmatize=lemmatize),
    }

def validate_required_columns(dataframe: pd.DataFrame) -> None:
    missing = [column for column in REQUIRED_COLUMNS if column not in dataframe.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

def preprocess_dataframe(dataframe: pd.DataFrame, *, lemmatize: bool = False) -> pd.DataFrame:
    """Validate the source schema and append the Phase 4 document fields."""
    validate_required_columns(dataframe)
    result = dataframe.copy()
    documents = result.apply(lambda row: preprocess_document(row["Title"], row["Abstract"], lemmatize=lemmatize), axis=1, result_type="expand")
    result[["title", "abstract", "clean_text"]] = documents[["title", "abstract", "clean_text"]]
    if "category" not in result.columns:
        result["category"] = result["Topic"].fillna("").astype(str).str.strip()
    return result

In [ ]:
# Load the merged local dataset. CSV files remain ignored by Git.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "processed_data" / "merged_research_papers.csv"

papers = pd.read_csv(DATA_PATH)
validate_required_columns(papers)
processed_papers = preprocess_dataframe(papers, lemmatize=False)

print(f"Loaded {len(papers):,} papers from {DATA_PATH}")
print(f"Required columns present: {set(REQUIRED_COLUMNS).issubset(processed_papers.columns)}")
processed_papers[["title", "abstract", "clean_text", "category"]].head(3)

In [ ]:
# Focused verification of the behavior requested in Phase 4.
example = preprocess_document(
    "Transformer Based Sentiment Analysis",
    "This paper proposes a BERT-based model for NLP on GPU hardware.",
)
assert example["title"] == "Transformer Based Sentiment Analysis"
assert "bert-based" in example["clean_text"]
assert all(term in example["clean_text"] for term in ["bert-based", "nlp", "gpu"])
assert " this " not in f" {example['clean_text']} "
assert set(["title", "abstract", "clean_text"]).issubset(processed_papers.columns)
assert len(processed_papers) == len(papers)
print("Verification passed.")
print(example)